# Project: Plan Your Trip with Kayak & Weather Data

## Objective
The goal of this project is to recommend the **top 5 cities in France** to visit and the **top 20 hotels**, based on the best weather forecast for the upcoming week.

## Workflow
1.  **Geolocation**: Retrieve GPS coordinates (latitude, longitude) for a list of target cities using the Nominatim API.
2.  **Weather Forecast**: Fetch 7-day weather forecasts for each city using the OpenWeatherMap API.
3.  **Scoring**: Calculate a "Weather Score" for each city based on temperature, rain probability, wind, etc., to identify the best destinations.
4.  **Accommodation**: Scrape Booking.com for the top hotels in the recommended cities.
5.  **Storage**: Save the cleaned data into an **Amazon S3** bucket and a **SQL Database** for further analysis or visualization.

In [ ]:
# Import libraries
import pandas as pd
import requests
import json
import time
import datetime
import uuid
import os
import plotly.express as px
from dotenv import load_dotenv
import boto3
from sqlalchemy import create_engine, text, Table, Column, Integer, String, MetaData, ForeignKey, Float, Date

# Load environment variables from .env file
load_dotenv()

# Configuration Variables
weather_api_secret = os.getenv('WEATHER_API_SECRET')
aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
bucket_name = os.getenv('BUCKET')

# Database Configuration
host = os.getenv('HOST_SQL_ALCHEMY')
port = os.getenv('PORT_SQL_ALCHEMY')
database = os.getenv('DATABASE_SQL_ALCHEMY')
user = os.getenv('USER_SQL_ALCHEMY')
password = os.getenv('PASSWORD_SQL_ALCHEMY')

In [ ]:
# Read initial list of cities from JSON
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

In [ ]:
# Create a working copy to avoid modifying the original dataframe
df = df_source.copy()

In [ ]:
# Define headers for API calls to mimic a real browser user-agent
# This helps avoid being blocked by some APIs.
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

# 1. Geolocation

In [ ]:
# Fetch coordinates for each city using Nominatim API
# We add a delay (time.sleep) to respect the API's usage policy and avoid rate limiting.

for index, row in df.iterrows():
    try:
        # API Call
        url = f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json"
        res = requests.get(url, headers=headers)
        
        if res.status_code == 200 and res.json():
            city_data = res.json()[0]
            df.loc[index, "lat"] = city_data["lat"]
            df.loc[index, "lon"] = city_data["lon"]
        else:
            print(f"Could not find coordinates for {row['city']}")
            
    except Exception as e:
        print(f"Error processing {row['city']}: {e}")
        
    time.sleep(1) # Pause to respect API rate limits

df.head()

# Checkpoint: Save result to CSV to avoid re-running expensive API calls
df.to_csv("cities_with_geoposition.csv", index=False)

In [ ]:
# [Optional] Reload data from CSV if restarting the notebook
if os.path.exists("cities_with_geoposition.csv"):
    df = pd.read_csv("cities_with_geoposition.csv")
    print("Loaded cities data from CSV.")
else:
    print("CSV not found, using data from memory.")
    
df.head()

# 2. Weather Forecast

In [ ]:
# Fetch 5-day/3-hour forecast data from OpenWeatherMap
list_weather_data = []

for index, row in df.iterrows():
    # Check if we have valid coordinates
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue

    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&exclude=current,minutely,hourly,alerts&appid={weather_api_secret}"
    res_weather = requests.get(url, headers=headers)
    res_weather_json = res_weather.json()
    
    # Process each forecast entry in the response
    if 'list' in res_weather_json:
        for res in res_weather_json['list']:
            weather_entry = {
                "city": row['city'],
                "lat": row['lat'],
                "lon": row['lon'],
                "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%Y-%m-%d'),
                "hour": datetime.datetime.fromtimestamp(res['dt']).strftime('%H:%M'),
                "temp": res['main']['temp'],
                "prob_rain": res.get('pop', 0), # Probability of precipitation (0-1)
                "volume_rain": res.get('rain', {}).get('3h', 0),
                "wind_speed": res['wind']['speed'],
                "perc_cloud": res['clouds']['all']
            }
            list_weather_data.append(weather_entry)
    
    time.sleep(1) # Pause to respect API rate limits

df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())

# Checkpoint: Save weather data
df_weather.to_csv("weather_forecast.csv", index=False)

In [ ]:
# [Optional] Reload weather data
if os.path.exists("weather_forecast.csv"):
    df_weather = pd.read_csv("weather_forecast.csv")
    print("Loaded weather data from CSV.")
df_weather.head()

In [ ]:
# Data Pre-processing
# Convert probability of rain to percentage (0-100)
# Convert wind speed from m/s to km/h
df_weather['prob_rain'] = df_weather['prob_rain'] * 100
df_weather['wind_speed'] = df_weather['wind_speed'] * 3.6
df_weather.head()

In [ ]:
# Aggregate daily data
# We group by city and date to get daily statistics (Mean/Max/Min)
df_weather_groupby = df_weather.groupby(['city', 'lat', 'lon', 'date']).agg({
    'temp': ['mean', 'min', 'max'], 
    'prob_rain': 'max', 
    'volume_rain': ['mean', 'max', 'sum'], 
    'wind_speed': 'max', 
    'perc_cloud': 'mean'
}).reset_index()

df_weather_groupby.head()

## Scoring Methodology

We calculate a satisfaction score (0-100) for each weather metric, where **100 is perfect** and **0 is poor**.

### 1. Normalization (0-100)
| Metric | Target | Penalty Calculation |
| :--- | :--- | :--- |
| **Temperature** | 25°C | -4 pts per degree deviation from 25°C |
| **Rain Probability** | 0% | 100 - (Probability %) |
| **Rain Volume** | 0mm | -5 pts per mm |
| **Wind Speed** | 0 km/h | 100 - (Speed in km/h) |
| **Cloudiness** | 0% | 100 - (Cloud %) |

### 2. Weighted Final Score
The final score is a weighted average of individual scores:
- **Temperature**: 30%
- **Rain Probability**: 20%
- **Rain Volume**: 30% (Heavy penalty for rain)
- **Wind**: 10%
- **Clouds**: 10%

In [ ]:
# 1. Calculate Individual Scores

# Temperature: Target 25°C
df_weather_groupby['score_temp'] = 100 - (abs(df_weather_groupby[('temp', 'max')] - 25) * 4)
df_weather_groupby['score_temp'] = df_weather_groupby['score_temp'].clip(lower=0)

# Rain Probability: 0% is best
df_weather_groupby['score_rain_prob'] = 100 - (df_weather_groupby[('prob_rain', 'max')])

# Rain Volume: 0mm is best
df_weather_groupby['score_rain_vol'] = 100 - (df_weather_groupby[('volume_rain', 'sum')] * 5)
df_weather_groupby['score_rain_vol'] = df_weather_groupby['score_rain_vol'].clip(lower=0)

# Wind Speed: 0 km/h is best
df_weather_groupby['score_wind'] = 100 - df_weather_groupby[('wind_speed', 'max')]
df_weather_groupby['score_wind'] = df_weather_groupby['score_wind'].clip(lower=0)

# Cloudiness: 0% is best
df_weather_groupby['score_cloud'] = 100 - df_weather_groupby[('perc_cloud', 'mean')]

# 2. Calculate Weighted Final Score
df_weather_groupby['total_score'] = (
    df_weather_groupby['score_temp'] * 0.3 +
    df_weather_groupby['score_rain_prob'] * 0.2 +
    df_weather_groupby['score_rain_vol'] * 0.3 +
    df_weather_groupby['score_wind'] * 0.1 +
    df_weather_groupby['score_cloud'] * 0.1
)

# 3. Identify Top 5 Cities (averaged over the 7-day period)
top_cities = df_weather_groupby.groupby(['city', 'lat', 'lon'])['total_score'].mean().sort_values(ascending=False)
print("--- FINAL RANKING ---")
print(top_cities.head(10))

# Save results
df_weather_groupby.to_csv("weather_forecast_with_score.csv", index=False)

In [ ]:
# Prepare DataFrame for visualization
df_top_cities = pd.DataFrame(top_cities.reset_index())
df_top_cities_top_10 = df_top_cities.iloc[0:10, :]  # Select top 10
df_top_cities_top_10

In [ ]:
# Visualize the top 10 cities based on weather score
fig = px.scatter_mapbox(
    df_top_cities_top_10, 
    lat="lat", 
    lon="lon",
    color="total_score",
    size="total_score", 
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15,
    zoom=4, 
    center={"lat": 46.2276, "lon": 2.2137}, # France Center
    mapbox_style="carto-positron", 
    hover_name="city",
    title="Top 10 Destinations in France (Weather Based)"
)

fig.show()

# 3. Accommodation (Booking.com)
We use a Scrapy spider to fetch hotel details for the top cities.

In [ ]:
# Remove previous file if exists
if os.path.exists('hotels.json'):
    os.remove('hotels.json')

# Run the scraper
!cd booking_scraper_project && scrapy crawl booking_spider -O ../hotels.json

## Hotel Analysis & Visualization

In [ ]:
# Load scraped hotel data
if os.path.exists("hotels.json"):
    df_hotels = pd.read_json("hotels.json")
    
    # Clean & format data
    df_hotels = df_hotels.dropna()
    
    # Convert score to float (replacing comma if necessary)
    df_hotels['score'] = df_hotels['score'].astype(str).str.replace(',', '.').astype(float)
    df_hotels = df_hotels.sort_values(by=["score"], ascending=False)
    
    display(df_hotels.head())
else:
    print("hotels.json not found. Make sure the scraper ran successfully.")

In [ ]:
# Visualize Top 20 Hotels in France
if 'df_hotels' in locals():
    fig = px.scatter_mapbox(
        df_hotels.head(20), 
        lat="lat", 
        lon="lng",
        color="score",
        size="score", 
        color_continuous_scale=px.colors.cyclical.IceFire,
        size_max=15,
        zoom=4, 
        center={"lat": 46.2276, "lon": 2.2137}, 
        mapbox_style="carto-positron", 
        hover_name="name",
        title="Top 20 Hotels in France"
    )
    fig.show()

# 4. Data Storage (S3 & SQL)
Prepare the dataframes and upload them to the cloud for persistence.

In [ ]:
# Data Formatting
# Assign UUIDs to cities and flatten the weather table for SQL compatibility

# 1. Add IDs to Cities
df_top_cities['id'] = [str(uuid.uuid4()) for _ in range(len(df_top_cities))]

# Save City Table
if not os.path.exists("final_output"):
    os.makedirs("final_output")
    
df_top_cities.rename(columns={'city': 'name'}).to_csv("final_output/villes_table.csv", index=False)

# 2. Flatten Weather Data (MultiIndex -> Single Level)
df_weather_flat = df_weather_groupby.reset_index()
df_weather_flat.columns = ['_'.join(c).strip('_') for c in df_weather_flat.columns.to_flat_index()]

# columns to keep
columns_weather = ['city', 'date', 'temp_mean', 'temp_min', 'temp_max', 
                   'prob_rain_max', 'volume_rain_mean', 'volume_rain_max', 'volume_rain_sum', 
                   'wind_speed_max', 'perc_cloud_mean', 'score_temp', 'score_rain_prob', 
                   'score_rain_vol', 'score_wind', 'score_cloud', 'total_score']
df_weather_flat = df_weather_flat[columns_weather]

# 3. Merge Function for ID linking
def merge_and_save(df_data, df_cities, filename):
    merged = df_data.merge(df_cities[['city', 'id']], on='city', how='left')
    merged = merged.rename(columns={'id': 'city_id'}).drop(columns=['city'])
    merged['id'] = [str(uuid.uuid4()) for _ in range(len(merged))]
    merged.to_csv(f"final_output/{filename}.csv", index=False)
    return merged

df_top_cities_light = df_top_cities[['id', 'city']]
df_weather_final = merge_and_save(df_weather_flat, df_top_cities_light, "weather_table")

if 'df_hotels' in locals():
    df_hotel_final = merge_and_save(df_hotels, df_top_cities_light, "hotels_table")

In [ ]:
# Amazon S3 Upload
try:
    session = boto3.Session(aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key)
    s3 = session.resource("s3")
    bucket = s3.create_bucket(Bucket=bucket_name)
    
    # Upload all CSV files in final_output
    files = [f for f in os.listdir("final_output") if f.endswith('.csv')]
    for file in files:
        s3.Bucket(bucket_name).upload_file(f"final_output/{file}", f"{file}")
        print(f"Uploaded {file} to S3.")
        
except Exception as e:
    print(f"S3 Upload Warning: {e}")

In [ ]:
# SQL Alchemy Setup
try:
    connection_string = f"postgresql+psycopg2://{user}:{password}@{host}/{database}"
    engine = create_engine(connection_string, echo=False)
    meta = MetaData()
except Exception as e:
    print(f"Database setup failed: {e}")
    engine = None

In [ ]:
if engine:
    # 1. Cities Table
    cities = Table(
        'cities', meta,
        Column('id', String, primary_key=True),
        Column('name', String),
        Column('lat', Float),
        Column('lon', Float),
        Column('total_score', Float)
    )
    
    # Create table and upload
    meta.create_all(engine)
    if 'df_top_cities' in locals():
        df_top_cities.rename(columns={'city': 'name'}).to_sql('cities', engine, if_exists='append', index=False)
        print("Cities uploaded to SQL.")

In [ ]:
if engine and 'df_weather_final' in locals():
    # 2. Weather Table
    weather = Table(
        'weather', meta,
        Column('id', String, primary_key=True),
        Column('date', Date),
        Column('temp_mean', Float),
        Column('temp_min', Float),
        Column('temp_max', Float),
        Column('prob_rain_max', Float),
        Column('volume_rain_mean', Float),
        Column('volume_rain_max', Float),
        Column('volume_rain_sum', Float),
        Column('wind_speed_max', Float),
        Column('perc_cloud_mean', Float),
        Column('score_temp', Float),
        Column('score_rain_prob', Float),
        Column('score_rain_vol', Float),
        Column('score_wind', Float),
        Column('score_cloud', Float),
        Column('total_score', Float),
        Column('city_id', String, ForeignKey('cities.id'))
    )
    
    meta.create_all(engine)
    
    # Ensure date format
    df_weather_final['date'] = pd.to_datetime(df_weather_final['date'])
    df_weather_final.to_sql('weather', engine, if_exists='append', index=False)
    print("Weather uploaded to SQL.")

In [ ]:
if engine and 'df_hotel_final' in locals():
    # 3. Hotels Table
    hotels = Table(
        'hotels', meta,
        Column('id', String, primary_key=True),
        Column('name', String),
        Column('url', String),
        Column('score', Float),
        Column('lat', Float),
        Column('lng', Float),
        Column('description', String),
        Column('city_id', String, ForeignKey('cities.id'))
    )
    
    meta.create_all(engine)
    df_hotel_final.to_sql('hotels', engine, if_exists='append', index=False)
    print("Hotels uploaded to SQL.")